This notebook trains two lightweight models to replace the RoBERTa-based teacher pipeline:

| Model | Task | Algorithm | Evaluated F1 |
|---|---|---|---|
| **Span Identifier (SI)** | Binary: does this sentence contain propaganda? | Logistic Regression + TF-IDF (word + char n-grams) | ~0.51 (PROP) |
| **Technique Classifier (TC)** | 14-class: which propaganda technique is used? | SGD / Modified-Huber SVM + TF-IDF (word + char n-grams) | ~0.38 macro F1 |

**Architecture rationale:**
- The dataset contains long news articles where propaganda spans are sparse (~7% of tokens). Predicting at the token level produces extreme class imbalance and poor generalisation.
- Splitting documents into sentences first reduces the task to a balanced binary classification problem (18% of sentences contain at least one propaganda span), which TF-IDF + LR handles effectively.
- Technique classification operates on the extracted span text only. Word + character n-grams capture both lexical patterns (specific phrases used in each technique) and morphological patterns (suffixes, prefixes) that distinguish techniques like *Loaded Language* from *Flag-Waving*.
- No neural networks, no GPU, no transformers — the full pipeline runs on CPU in under 3 minutes.

In [1]:
import ast
import re
import collections
import warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

In [2]:
BASE_DIR   = Path.cwd().resolve().parent
DATA_PATH  = BASE_DIR / 'data' / 'processed' / 'distilled.csv'
GOLD_DATA_PATH  = BASE_DIR / 'data' / 'processed' / 'gold_distilled.csv'
MODELS_DIR = BASE_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

SI_PATH = MODELS_DIR / 'distilled_si.joblib'
TC_PATH = MODELS_DIR / 'distilled_tc.joblib'

RANDOM_STATE = 42
TEST_SIZE    = 0.20

print(f'Data: {DATA_PATH}')
print(f'SI out: {SI_PATH}')
print(f'TC out: {TC_PATH}')

Data: /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/data/processed/distilled.csv
SI out: /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/distilled_si.joblib
TC out: /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/distilled_tc.joblib


In [3]:
#Mapping for consistency and class balancing
TECHNIQUE_MAP = {
    "Red_Herring": "Whataboutism_Straw_Men_Red_Herring",
    "Minimisation": "Exaggeration_Minimisation",
    "Causal_Oversimplification": "Oversimplification_Black-and-White", #New combined name
    "Black-and-White_Fallacy": "Oversimplification_Black-and-White",   #New combined name
    "Slogans": "Slogans_Thought-terminating_Cliches",                  #New combined name
    "Thought-terminating_Cliches": "Slogans_Thought-terminating_Cliches" #New combined name
}

In [4]:
TC_CLASSES = [
    "Doubt",
    "Appeal_to_Authority",
    "Repetition",
    "Appeal_to_fear-prejudice",
    "Oversimplification_Black-and-White",
    "Slogans_Thought-terminating_Cliches",
    "Loaded_Language",
    "Flag-Waving",
    "Name_Calling_Labeling",
    "Whataboutism_Straw_Men_Red_Herring",
    "Exaggeration_Minimisation",
    "Bandwagon_Reductio_ad_hitlerum"
]

In [5]:
#Load and parse data
def get_technique(item: dict) -> str:
    """Extract technique label and map to standard/merged classes."""
    tech = None
    if 'technique' in item:
        tech = item['technique']
    else:
        for k, v in item.items():
            if k != 'span':
                tech = v
                break
    return TECHNIQUE_MAP.get(tech, tech)


In [6]:
#Load enhanced dataset with some SemEval examples, some from news articles run through our models, and some generated as examples by Gemini using the official SemEval definitions for each technique
train_df = pd.read_csv(DATA_PATH)
train_df = train_df.reset_index(drop=True)
train_df.head()

,Unnamed: 0,text,propaganda
0,0,Outrage as Donald Trump suggests injecting dis...,"[{'span': [0, 6], 'technique': 'Loaded_Languag..."
1,1,The senator's vile betrayal of working familie...,"[{'span': [14, 18], 'technique': 'Loaded_Langu..."
2,2,Brave freedom fighters resist the tyrannical o...,"[{'span': [0, 5], 'technique': 'Loaded_Languag..."
3,3,The corrupt elites are bleeding this country d...,"[{'span': [4, 10], 'technique': 'Loaded_Langua..."
4,4,A catastrophic failure of leadership has plung...,"[{'span': [2, 13], 'technique': 'Loaded_Langua..."


In [7]:
#Load gold spans, these are examples from the official SemEval competition, so we know for sure the answers are correct
test_df = pd.read_csv(GOLD_DATA_PATH)
test_df = test_df.reset_index(drop=True)
test_df.head()

,Unnamed: 0,text,propaganda
0,234,"President Donald Trump Proposes ""Simple Immigr...","[{'span': [59, 79], 'technique': 'Slogans'}, {..."
1,157,ICE arrests 20 in Kansas City during 4-day ope...,"[{'span': [2216, 2270], 'technique': 'Repetiti..."
2,82,The Dictator Pope: A Call to Hierarchical Oppo...,"[{'span': [0, 17], 'technique': 'Name_Calling_..."
3,105,Man who sold ammo to Las Vegas gunman speaks o...,"[{'span': [2296, 2312], 'technique': 'Loaded_L..."
4,183,WHO Prepares For “Worst Case” As Congo Ebola O...,"[{'span': [3936, 3945], 'technique': 'Loaded_L..."


In [8]:
#Parse the 'propaganda' column from string → list of dicts
train_df['propaganda_parsed'] = train_df['propaganda'].apply(ast.literal_eval)
test_df['propaganda_parsed'] = test_df['propaganda'].apply(ast.literal_eval)

print(f'Train rows loaded: {len(train_df):,}')
print(f'Columns: {train_df.columns.tolist()}')
print(f'Test rows loaded: {len(test_df):,}')
print(f'Columns: {test_df.columns.tolist()}')

Train rows loaded: 6,219
Columns: ['Unnamed: 0', 'text', 'propaganda', 'propaganda_parsed']
Test rows loaded: 72
Columns: ['Unnamed: 0', 'text', 'propaganda', 'propaganda_parsed']


In [9]:
#Summarise techniques
all_techniques = []
for entries in train_df['propaganda_parsed']:
    for item in entries:
        t = get_technique(item)
        if t:
            all_techniques.append(t)

tech_counts = collections.Counter(all_techniques)
print(f'Total span annotations: {len(all_techniques):,}')
print(f'Unique techniques: {len(tech_counts)}')
for tech, cnt in tech_counts.most_common():
    print(f'{tech:<45s} {cnt:>6,}')

Total span annotations: 29,364
Unique techniques: 12
Bandwagon_Reductio_ad_hitlerum                 6,226
Loaded_Language                                4,442
Whataboutism_Straw_Men_Red_Herring             2,914
Flag-Waving                                    2,494
Name_Calling_Labeling                          2,264
Appeal_to_Authority                            2,040
Doubt                                          1,954
Repetition                                     1,786
Appeal_to_fear-prejudice                       1,756
Exaggeration_Minimisation                      1,526
Oversimplification_Black-and-White             1,172
Slogans_Thought-terminating_Cliches              790


In [10]:
#Feature engineering
def split_sentences(text: str):
    """Split text into sentences, returning (sentence_str, start_char, end_char).

    Uses a simple regex that splits on sentence-ending punctuation. This is
    intentionally lightweight — the goal is reasonable chunks, not perfect
    linguistic sentences.
    """
    sents = []
    for m in re.finditer(r'[^.!?\n]+[.!?\n]+|[^.!?\n]+$', text):
        s = m.group().strip()
        if s:
            sents.append((s, m.start(), m.end()))
    return sents

In [11]:
def build_sentence_dataset(dataframe: pd.DataFrame):
    """Build sentence-level binary dataset for Span Identification.

    Returns
    -------
    sentences : list[str]
    labels    : list[int]  — 1 if sentence contains ≥1 propaganda span, else 0
    """
    sentences, labels = [], []
    for _, row in dataframe.iterrows():
        text    = row['text']
        entries = row['propaganda_parsed']
        sents   = split_sentences(text)

        #Collect all char-level prop ranges for this document
        prop_ranges = [(item['span'][0], item['span'][1]) for item in entries]

        for sent, ss, se in sents:
            if len(sent.split()) < 3:  #skip trivially short fragments
                continue
            has_prop = any(ps < se and pe > ss for ps, pe in prop_ranges)
            sentences.append(sent)
            labels.append(1 if has_prop else 0)

    return sentences, labels

In [12]:
def build_span_dataset(dataframe: pd.DataFrame, min_count: int = 5):
    """Build span-level multi-class dataset for Technique Classification.

    Parameters
    ----------
    min_count : drop technique classes with fewer than this many examples

    Returns
    -------
    span_texts : list[str]
    techniques : list[str]
    classes    : list[str]  — sorted list of retained class names
    """
    span_texts, techniques = [], []
    for _, row in dataframe.iterrows():
        text = row['text']
        sents = split_sentences(text)
        entries = row['propaganda_parsed']

        for item in entries:
            tech = get_technique(item)
            s, e = item['span']

            #Find the sentence that contains this span
            context_sent = ""
            for sent_str, ss, se in sents:
                if ss <= s and se >= e:
                    context_sent = sent_str
                    break

            combined_input = f"SPAN: {text[s:e].strip()} CONTEXT: {context_sent}"
            if text[s:e].strip() and tech:
                span_texts.append(combined_input)
                techniques.append(tech)

    #Drop rare classes
    counts = collections.Counter(techniques)
    rare   = {k for k, v in counts.items() if v < min_count}
    keep   = [(t, y) for t, y in zip(span_texts, techniques) if y not in rare]
    span_texts = [x[0] for x in keep]
    techniques = [x[1] for x in keep]
    classes    = sorted(set(techniques))

    return span_texts, techniques, classes

In [13]:
#Handcrafted linguistic features
#Each set targets words strongly associated with a specific propaganda signal.

_POSITIVE_WORDS = frozenset([
    'good', 'great', 'excellent', 'wonderful', 'amazing', 'fantastic',
    'brilliant', 'heroic', 'noble', 'brave', 'righteous', 'glorious',
    'proud', 'victory', 'success', 'freedom', 'liberty', 'justice',
    'beautiful', 'perfect', 'best', 'finest', 'superior', 'outstanding',
    'magnificent', 'exceptional', 'blessed', 'sacred', 'holy', 'pure',
    'strong', 'powerful', 'winning', 'triumph', 'love', 'hope', 'unity',
])

_NEGATIVE_WORDS = frozenset([
    'bad', 'evil', 'terrible', 'horrible', 'disgusting', 'corrupt', 'vile',
    'wicked', 'traitor', 'criminal', 'thug', 'monster', 'liar', 'crook',
    'dangerous', 'threat', 'destroy', 'destruction', 'collapse', 'fail',
    'failure', 'wrong', 'false', 'lie', 'fraud', 'coward', 'puppet',
    'oppressive', 'tyrannical', 'shame', 'shameful', 'outrageous', 'hate',
    'enemy', 'enemies', 'terror', 'catastrophe', 'disaster', 'worst',
    'stupid', 'idiot', 'moron', 'loser', 'radical', 'extremist',
])

_INTENSIFIERS = frozenset([
    'very', 'extremely', 'absolutely', 'incredibly', 'utterly', 'totally',
    'completely', 'entirely', 'truly', 'deeply', 'highly', 'strongly',
    'tremendously', 'enormously', 'terribly', 'awfully', 'dreadfully',
])

_FEAR_WORDS = frozenset([
    'danger', 'dangerous', 'threat', 'threatening', 'risk', 'crisis',
    'catastrophe', 'catastrophic', 'disaster', 'disastrous', 'destroy',
    'destruction', 'collapse', 'invasion', 'attack', 'terror', 'terrorism',
    'fear', 'afraid', 'panic', 'alarm', 'horror', 'dread', 'nightmare',
    'enemy', 'enemies', 'menace', 'peril', 'urgent', 'emergency',
    'ebola', 'plague', 'outbreak'
])

_PATRIOTIC_WORDS = frozenset([
    'nation', 'national', 'country', 'homeland', 'freedom', 'liberty',
    'patriot', 'patriotic', 'patriotism', 'american', 'america', 'citizens',
    'people', 'great', 'proud', 'pride', 'tradition', 'values', 'heritage',
    'sovereignty', 'defend', 'protect', 'glory',
])

_AUTHORITY_WORDS = frozenset([
    'expert', 'experts', 'scientist', 'scientists', 'study', 'studies',
    'research', 'researchers', 'professor', 'doctor', 'dr', 'phd',
    'evidence', 'proven', 'facts', 'data', 'according', 'report', 'reports',
    'official', 'government', 'university', 'institute', 'organization',
])


_DOUBT_WORDS = frozenset([
    'really', 'truly', 'actually', 'allegedly', 'supposedly', 'claimed',
    'question', 'doubt', 'suspicious', 'wonder', 'whether', 'failed',
    'failing', 'lied', 'lies', 'lie', 'misleading', 'wrong', 'incorrect',
    'false', 'cover', 'hide', 'hoax', 'conspiracy', 'fraud', 'suspicious'
])

_LOADED_WORDS = frozenset([
    'radical', 'extremist', 'traitor', 'corrupt', 'evil', 'wicked', 'vile',
    'disgusting', 'outrageous', 'shameful', 'coward', 'monster',
    'oppressive', 'tyrannical', 'fascist', 'socialist', 'communist',
    'heroic', 'noble', 'brave', 'righteous', 'brilliant', 'glorious',
    'sacred', 'holy', 'pure', 'blessed', 'cursed', 'infidel', 'regime',
    'puppet', 'globalist', 'elite', 'treacherous', 'hardworking',
])

_NAME_CALL_WORDS = frozenset([
    'idiot', 'moron', 'fool', 'loser', 'liar', 'crook', 'thug', 'criminal',
    'terrorist', 'coward', 'hypocrite', 'puppet', 'stooge', 'clown',
    'elitist', 'snowflake', 'racist', 'bigot', 'extremist', 'deplorable',
])

_COMMON_PHRASES = [
    #Loaded / fear phrases
    'wake up', 'open your eyes', 'the truth is', 'they want you to',
    'mainstream media', 'fake news', 'the real agenda', 'hidden agenda',
    'working class', 'ordinary people', 'our way of life',
    #Flag-waving phrases
    'our nation', 'our country', 'our people', 'our values', 'our freedom',
    'stand up for', 'fight for', 'true patriot', 'defend our',
    #Authority / doubt phrases
    'according to', 'studies show', 'experts say', 'it has been proven',
    'there is no evidence', 'no one talks about', 'nobody mentions',
    #Black-and-white / slogans
    'either you', 'with us or', 'you are either', 'there is no other',
    'only choice', 'the only way',
]

_BANDWAGON_HITLERUM_WORDS = frozenset([
    'everyone', 'everybody', 'millions', 'nobody', 'join', 'consensus',
    'popular', 'majority', 'hitler', 'nazi', 'fascist', 'reminiscent',
    'history', 'repeating', 'concentration', 'genocide', 'dictator',
    'gestapo', 'third', 'reich', 'totalitarian', 'propaganda',
])

_LOGICAL_FALLACY_WORDS = frozenset([
    'whatabout', 'what about', 'instead', 'ignore', 'anyway', 'regardless', 'distraction',
    'argument', 'point', 'actually', 'strawman', 'focus', 'deflection',
    'excuse', 'side', 'issue', 'topic', 'irrelevant', 'meanwhile',
    'incidentally', 'divert', 'aside', 'besides',
])

In [14]:
def compute_handcrafted_features(texts):
    """Compute dense handcrafted linguistic features for a list of texts.

    All features are computed with stdlib + numpy — no extra dependencies.
    Every feature is scaled to [0, 1] to match TF-IDF sublinear_tf range,
    preventing raw counts from dominating the regularization signal.

    Features (17 total):
        0   polarity           (pos_hits - neg_hits) / n_words  in [-1, 1]
        1   subjectivity       (pos_hits + neg_hits) / n_words  in [ 0, 1]
        2   has_exclamation    binary: text contains '!'
        3   exclamation_rate   count('!') / n_words             in [ 0, 1]
        4   has_question       binary: text contains '?'
        5   caps_word_ratio    ALL-CAPS tokens (len≥2) / n_words in [ 0, 1]
        6   intensifier_rate   intensifier hits / n_words
        7   fear_rate          fear/threat word hits / n_words
        8   patriotic_rate     patriotic word hits / n_words
        9   authority_rate     authority word hits / n_words
        10  doubt_rate         doubt/discredit word hits / n_words
        11  loaded_rate        emotionally loaded word hits / n_words
        12  name_call_rate     name-calling word hits / n_words
        13  first_person_rate  we/our/us hits / n_words
        14  second_person_rate you/your hits / n_words
        15  phrase_rate        matched phrases / total phrases (fraction hit)
        16  length_norm        log1p(n_words) / log1p(300) — approx [0, 1]
        17  bandwagon_hitler_rate   bandwagon/ad-hitlerum hits / n_words
        18  fallacy_red_herring_rate whataboutism/red-herring hits / n_words

    Returns scipy.sparse.csr_matrix of shape (len(texts), 17).
    """
    n = len(texts)
    feat = np.zeros((n, 19), dtype=np.float32)
    n_phrases = max(len(_COMMON_PHRASES), 1)

    for i, text in enumerate(texts):
        tokens = re.findall(r"[A-Za-z']+", text)
        words_lower = [t.lower() for t in tokens]
        n_words = max(len(tokens), 1)
        text_lower = text.lower()

        # 0-1: lexicon-based sentiment (already ratios)
        pos_hits = sum(1 for w in words_lower if w in _POSITIVE_WORDS)
        neg_hits = sum(1 for w in words_lower if w in _NEGATIVE_WORDS)
        feat[i, 0] = np.clip((pos_hits - neg_hits) / n_words, -1.0, 1.0)
        feat[i, 1] = np.clip((pos_hits + neg_hits) / n_words,  0.0, 1.0)

        # 2-4: punctuation signals
        feat[i, 2] = 1.0 if '!' in text else 0.0
        feat[i, 3] = text.count('!') / n_words          # rate, not raw count
        feat[i, 4] = 1.0 if '?' in text else 0.0

        # 5: ALL-CAPS ratio (already a ratio)
        caps_count = sum(1 for t in tokens if len(t) >= 2 and t.isupper())
        feat[i, 5] = caps_count / n_words

        # 6-14, 17-18: lexicon hits — all normalised to per-word rates
        feat[i, 6]  = sum(1 for w in words_lower if w in _INTENSIFIERS)  / n_words
        feat[i, 7]  = sum(1 for w in words_lower if w in _FEAR_WORDS)     / n_words
        feat[i, 8]  = sum(1 for w in words_lower if w in _PATRIOTIC_WORDS)/ n_words
        feat[i, 9]  = sum(1 for w in words_lower if w in _AUTHORITY_WORDS)/ n_words
        feat[i, 10] = sum(1 for w in words_lower if w in _DOUBT_WORDS)    / n_words
        feat[i, 11] = sum(1 for w in words_lower if w in _LOADED_WORDS)   / n_words
        feat[i, 12] = sum(1 for w in words_lower if w in _NAME_CALL_WORDS)/ n_words
        feat[i, 13] = sum(1 for w in words_lower if w in
                          ('we', 'our', 'us', 'ourselves'))                / n_words
        feat[i, 14] = sum(1 for w in words_lower if w in
                          ('you', 'your', 'yourself', 'yourselves'))       / n_words
        feat[i, 17] = sum(1 for w in words_lower if w in _BANDWAGON_HITLERUM_WORDS) / n_words
        feat[i, 18] = sum(1 for w in words_lower if w in _LOGICAL_FALLACY_WORDS) / n_words

        # 15: fraction of known propaganda phrases that appear in this text
        feat[i, 15] = sum(1 for ph in _COMMON_PHRASES if ph in text_lower) / n_phrases

        # 16: log-length normalised to approximately [0, 1]
        feat[i, 16] = np.log1p(n_words) / np.log1p(300)

    return csr_matrix(feat)

In [15]:
print('Building sentence-level dataset...')
si_train_X_raw, si_train_y = build_sentence_dataset(train_df)
si_test_X_raw,  si_test_y  = build_sentence_dataset(test_df)

print(f'Train sentences total: {len(si_train_X_raw):,}')
print(f'Prop sentences: {sum(si_train_y):,} ({100*np.mean(si_train_y):.1f}%)')
print()
print(f'Test sentences total: {len(si_test_X_raw):,}')
print(f'Prop sentences: {sum(si_test_y):,} ({100*np.mean(si_test_y):.1f}%)')

Building sentence-level dataset...
Train sentences total: 101,487
Prop sentences: 20,344 (20.0%)

Test sentences total: 3,420
Prop sentences: 1,049 (30.7%)


In [16]:
print('Building span-level dataset ...')
tc_train_X_raw, tc_train_y, _ = build_span_dataset(train_df, min_count=5)
tc_test_X_raw,  tc_test_y,  _ = build_span_dataset(test_df, min_count=5)

print(f'Training span samples: {len(tc_train_X_raw):,}')
print(f'Testing span samples: {len(tc_test_X_raw):,}')

Building span-level dataset ...
Training span samples: 29,364
Testing span samples: 1,330


In [17]:
si_train_sents, si_train_y = build_sentence_dataset(train_df)
si_test_sents,  si_test_y  = build_sentence_dataset(test_df)

print(f'SI train: {len(si_train_sents):,} sentences, {np.mean(si_train_y):.1%} prop')
print(f'SI test: {len(si_test_sents):,} sentences, {np.mean(si_test_y):.1%} prop')

SI train: 101,487 sentences, 20.0% prop
SI test: 3,420 sentences, 30.7% prop


In [18]:
#Model 1: Span Identifier
print('Vectorizing SI features...')

si_word_vec = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=20_000,
    sublinear_tf=True,
    analyzer='word',
)
si_char_vec = TfidfVectorizer(
    ngram_range=(2, 5),
    max_features=20_000,
    sublinear_tf=True,
    analyzer='char_wb',
)

si_X_train = hstack([
    si_word_vec.fit_transform(si_train_X_raw),
    si_char_vec.fit_transform(si_train_X_raw),
    compute_handcrafted_features(si_train_X_raw),
])
si_X_test = hstack([
    si_word_vec.transform(si_test_X_raw),
    si_char_vec.transform(si_test_X_raw),
    compute_handcrafted_features(si_test_X_raw),
])

print(f'SI feature matrix shape: {si_X_train.shape}')

Vectorizing SI features...
SI feature matrix shape: (101487, 40019)


In [19]:
print('Training Span Identifier...')

si_clf = LogisticRegression(
    C=1.0,
    max_iter=1000,
    class_weight='balanced',
    solver='lbfgs',
    random_state=RANDOM_STATE,
)
si_clf.fit(si_X_train, si_train_y)

print('Done.')

Training Span Identifier...
Done.


In [20]:
si_proba = si_clf.predict_proba(si_X_test)[:, 1]
preds = (si_proba >= 0.5).astype(int)
f1 = f1_score(si_test_y, preds, zero_division=0)
print(f1)

print(classification_report(
    si_test_y, preds,
    target_names=['Non-propaganda', 'Propaganda'],
    zero_division=0,
))

0.595814526056627
                precision    recall  f1-score   support

Non-propaganda       0.84      0.72      0.78      2371
    Propaganda       0.52      0.69      0.60      1049

      accuracy                           0.71      3420
     macro avg       0.68      0.71      0.69      3420
  weighted avg       0.74      0.71      0.72      3420



In [21]:
#Model 2: Technique Classifier
print('Vectorizing TC features ...')

tc_word_vec = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=30_000,
    sublinear_tf=True,
    analyzer='word',
)
tc_char_vec = TfidfVectorizer(
    ngram_range=(3, 6),
    max_features=20_000,
    sublinear_tf=True,
    analyzer='char_wb',
)

tc_X_train_v = hstack([
    tc_word_vec.fit_transform(tc_train_X_raw),
    tc_char_vec.fit_transform(tc_train_X_raw),
    compute_handcrafted_features(tc_train_X_raw),
])
tc_X_test_v = hstack([
    tc_word_vec.transform(tc_test_X_raw),
    tc_char_vec.transform(tc_test_X_raw),
    compute_handcrafted_features(tc_test_X_raw),
])

print(f'TC feature matrix shape: {tc_X_train_v.shape}')

Vectorizing TC features ...
TC feature matrix shape: (29364, 50019)


In [22]:
print('Training Technique Classifier...')

tc_clf = SGDClassifier(
    loss='modified_huber',
    alpha=1e-4,
    max_iter=200,
    class_weight='balanced',
    random_state=RANDOM_STATE,
)
tc_clf.fit(tc_X_train_v, tc_train_y)

print('Done.')

Training Technique Classifier...
Done.


In [23]:
tc_preds = tc_clf.predict(tc_X_test_v)
tc_macro = f1_score(tc_test_y, tc_preds, average='macro', zero_division=0)
tc_weighted = f1_score(tc_test_y, tc_preds, average='weighted', zero_division=0)

print(f'Macro F1: {tc_macro:.4f}')
print(f'Weighted F1: {tc_weighted:.4f}')
print(classification_report(tc_test_y, tc_preds, labels=TC_CLASSES, zero_division=0,))

Macro F1: 0.2897
Weighted F1: 0.3667
                                     precision    recall  f1-score   support

                              Doubt       0.33      0.50      0.40       101
                Appeal_to_Authority       0.07      0.19      0.10        31
                         Repetition       0.26      0.11      0.15       121
           Appeal_to_fear-prejudice       0.23      0.49      0.31        57
 Oversimplification_Black-and-White       0.25      0.34      0.29        76
Slogans_Thought-terminating_Cliches       0.32      0.43      0.37        46
                    Loaded_Language       0.61      0.32      0.42       464
                        Flag-Waving       0.30      0.64      0.41        53
              Name_Calling_Labeling       0.55      0.46      0.50       214
 Whataboutism_Straw_Men_Red_Herring       0.06      0.08      0.07        24
          Exaggeration_Minimisation       0.31      0.35      0.33       124
     Bandwagon_Reductio_ad_hitlerum   

In [24]:
#Save models
si_bundle = {
    'word_vectorizer': si_word_vec,
    'char_vectorizer': si_char_vec,
    'feature_extractor': compute_handcrafted_features,
    'classifier': si_clf,
    'threshold': 0.7,
    'description': 'Sentence-level propaganda span identifier.'
                   'Input: sentence string. Output: P(propaganda).'
}

tc_bundle = {
    'word_vectorizer': tc_word_vec,
    'char_vectorizer': tc_char_vec,
    'feature_extractor': compute_handcrafted_features,
    'classifier': tc_clf,
    'classes': TC_CLASSES,
    'description': 'Span-level technique classifier.'
                   'Input: span text string. Output: technique label.'
}

joblib.dump(si_bundle, SI_PATH)
joblib.dump(tc_bundle, TC_PATH)

print(f'SI model saved → {SI_PATH}')
print(f'TC model saved → {TC_PATH}')

SI model saved → /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/distilled_si.joblib
TC model saved → /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/distilled_tc.joblib


In [25]:
#How to load the models and run inference on a new article
si = joblib.load(SI_PATH)
tc = joblib.load(TC_PATH)


def predict_propaganda(article_text: str,
                        si_bundle: dict,
                        tc_bundle: dict) -> list[dict]:
    """Run the full propaganda detection pipeline on a raw article.

    Steps
    -----
    1. Split article into sentences.
    2. Score each sentence with the Span Identifier.
    3. For sentences above the detection threshold, classify the technique.

    Returns
    -------
    List of dicts, each with keys:
        'sentence'  : str   — the flagged sentence
        'span'      : [int, int]  — [start_char, end_char] in the article
        'technique' : str   — predicted propaganda technique
        'si_score'  : float — span identifier confidence (0–1)
    """
    sents = split_sentences(article_text)
    if not sents:
        return []

    sent_strings = [s for s, _, _ in sents]

    #Identify spans
    si_hc = si_bundle['feature_extractor'](sent_strings)
    si_feats = hstack([
        si_bundle['word_vectorizer'].transform(sent_strings),
        si_bundle['char_vectorizer'].transform(sent_strings),
        si_hc,
    ])
    si_scores  = si_bundle['classifier'].predict_proba(si_feats)[:, 1]
    threshold  = si_bundle['threshold']
    is_prop    = si_scores >= threshold

    flagged = [
        (sent_strings[i], sents[i][1], sents[i][2], float(si_scores[i]))
        for i in range(len(sents))
        if is_prop[i]
    ]

    if not flagged:
        return []

    #Classify techniques
    flagged_texts = [f[0] for f in flagged]
    tc_hc = tc_bundle['feature_extractor'](flagged_texts)
    tc_feats = hstack([
        tc_bundle['word_vectorizer'].transform(flagged_texts),
        tc_bundle['char_vectorizer'].transform(flagged_texts),
        tc_hc,
    ])
    techniques = tc_bundle['classifier'].predict(tc_feats)

    results = []
    for (sent, ss, se, score), tech in zip(flagged, techniques):
        results.append({
            'sentence': sent,
            'span': [ss, se],
            'technique': tech,
            'si_score': round(score, 3),
        })

    return results

In [26]:
#Example
DEMO_ARTICLE = """
The treacherous elites are draining our nation dry while honest, hardworking
patriots watch helplessly from the sidelines. Either you stand with us or you
stand against everything we hold dear. The radical agenda pushed by these
globalist puppets must be exposed for what it truly is. Meanwhile, scientists
released a new study on agricultural yields in Southeast Asia. The heroic
resistance fighters continue their noble struggle for freedom and justice.
"""

results = predict_propaganda(DEMO_ARTICLE, si, tc)

print(f'Detected {len(results)} propaganda span(s):\n')
for r in results:
    print(f"• [{r['technique']}] (score={r['si_score']})")
    print(f"  Sentence: {r['sentence'][:120]}")

Detected 6 propaganda span(s):

• [Loaded_Language] (score=0.984)
  Sentence: The treacherous elites are draining our nation dry while honest, hardworking
• [Oversimplification_Black-and-White] (score=0.792)
  Sentence: Either you stand with us or you
• [Name_Calling_Labeling] (score=0.718)
  Sentence: The radical agenda pushed by these
• [Name_Calling_Labeling] (score=0.944)
  Sentence: globalist puppets must be exposed for what it truly is.
• [Loaded_Language] (score=0.991)
  Sentence: The heroic
• [Loaded_Language] (score=0.811)
  Sentence: resistance fighters continue their noble struggle for freedom and justice.
